# Import library

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
from collections import Counter
import cv2
from glob import glob
from tqdm import tqdm
from termcolor import colored

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Custom dataset class

In [ ]:
class CustomVOCDataset(torchvision.datasets.VOCDetection):
    def __init__(self, *args, class_mapping, S=7, B=2, C=20, custom_transforms=None, **kwargs):
        super(CustomVOCDataset, self).__init__(*args, **kwargs)
        # initialize YOLO-specific coniguration params
        self.S = S # grid size s x s
        self.B = B # number of bounding boxes
        self.C = C # number of classes
        self.class_mapping = class_mapping # mapping of class names to class indices
        self.custom_transforms = custom_transforms

    def __getitem__(self, index):
        # get an image and its target (annotation) from the VOC dataset
        image, target = super(CustomVOCDataset, self).__getitem__(index)
        img_width, img_height = image.size

        # convert target annotations to YOLO format bounding boxes
        boxes = convert_to_yolo_format(target, img_width, img_height, self.class_mapping)

        just_boxes = boxes[:, 1:]
        labels = boxes[:, 0]

        # transform
        if self.custom_transforms:
            sample = {
                'image': np.array(image),
                'bboxes': just_boxes,
                'labels': labels
            }
            sample = self.custom_transforms(**sample)
            image = sample['image']
            boxes = sample['bboxes']
            labels = sample['labels']

        # create an empty label matrix for YOLO ground truth
        label_matrix = torch.zeros((self.S, self.S, self.C + 5 * self.B))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.float32)
        image = torch.tensor(image, dtype=torch.float32)

        # iterate through each bounding box in YOLO format
        for box, class_label in zip (boxes, labels):
            x, y, width, height = box.tolist()
            class_label = int(class_label)

            # calculate the grid cell (i, j) that this box belongs to
            i, j = int(self.S * y), int(self.S * x)
            x_cell, y_cell = self.S * x - j, self.S * y - i

            # calculate the width and height of the box relative to the grid cell
            width_cell, height_cell = (
                width * self.S,
                height * self.S,
            )

            # if no object has been found in this specific cell (i, j) before:
            if label_matrix[i, j, 20] == 0:
                # mark that an object exists in this cell
                label_matrix[i, j, 20] = 1

                # store the box coordinates as an offset from the cell boundaries:
                box_coordinates = torch.tensor(
                    [x_cell, y_cell, width_cell, height_cell]
                )

                # set the box coordinates in the label matrix
                label_matrix[i, j, 21:25] = box_coordinates

                # set the one-hot encoding for the class label
                label_matrix[i, j, class_label] = 1

        return image, label_matrix

# Other utility functions

In [7]:
def convert_to_yolo_format(target, img_width, img_height, class_mapping):
    """
    convert annotation data from VOC format to YOLO format

    params:
    target (dict): annotation data from VOCDetection dataset
    img_width (int): width of the original image
    img_height (int): height of the original image
    class_mapping (dict): mapping from class names to integer IDs
    
    returns:
    torch.Tensor: tensor of shape [N, 5] for N bounding boxes, each with [class_id, x_center, y_center, width, height]
    """

    # extract the list of annotations from the target dictionary
    annotations = target['annotation']['object']

    # get the real width and height of the image from the annotation
    real_width = int(target['annotation']['size']['width'])
    real_height = int(target['annotation']['size']['height'])

    # ensure the annotations is a list, even if there's only one object
    if not isinstance(annotations, list):
        annotations = [annotations]

    # initialize an empty list to store the converted bounding boxes
    boxes = []

    # loop through each annotation and convert it to YOLO format
    for anno in annotations:
        xmin = int(anno['bndbox']['xmin']) / real_width
        xmax = int(anno['bndbox']['xmax']) / real_width
        ymin = int(anno['bndbox']['ymin']) / real_height
        ymax = int(anno['bndbox']['ymax']) / real_height

        # calcualate the center coordinates, widths and height of the bounding box
        x_center = (xmin + xmax) / 2
        y_center = (ymin + ymax) / 2
        width = xmax - xmin
        height = ymax - ymin

        # retrieve the class name from the annotation and map it to an integer ID
        class_name = anno['name']
        class_id = class_mapping[class_name] if class_name in class_mapping else 0

        # append the YOLO formatted bounding box to the list
        boxes.append([class_id, x_center, y_center, width, height])

    # convert the list of boxes to a torch tensor
    return np.array(boxes)

In [8]:
def intersection_over_union(boxes_preds, boxes_labels, box_format="midpoint"):
    """
    calcualate the intersection over union (IoU) between bounding boxes

    params:
        bboxes_preds (tensor): predicted bounding boxes (BATCH_SIZE, 4)
        boxes_labels (tensor): ground truth bounding boxes (BATCH_SIZE, 4)
        box_format (str): Box format, can be "mid point" or "corners"

    returns:
        tensor: intersection over union scores for each example
    """

    # check if the box format is "mid point"
    if box_format == "midpoint":
        # calculate coordinates of top-left (x1, y1) and bottom-right (x2, y2) points for predicted boxes
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2

        # calculate coordinates of top-left (x1, y1) and bottom-right (x2, y2) points for ground truth boxes
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    # check if the box format is "corners"
    if box_format == "corners":
        # extract coordinates for predicted boxes
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]

        # extract coordinates for ground truth boxes
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    # calculate coordinates of the intersection rectangle
    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # compute the area of the intersection rectangle, clamp(0) to handle cases where they do not overlap
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    # calculate the areas of the predicted and ground truth boxes
    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    # calcualte the intersection ober union, adding small epsilon to avoid division by zero
    return intersection / (box1_area + box2_area - intersection + 1e-6)


In [9]:
def non_max_suppresion(bboxes, iou_threshold, threshold, box_format="corners"):
    """
    perform non-maximum supression on a list of bounding boxes
    params:
        bboxes (list): list of bounding boxes, each representedd as [class_pred, probability_score, x1, y1, x2, y2]
        iou_threshold (float): IoU threshold to determine correct predicted bounding boxes
        threshold (float): threshold to discard predicted bounding boxes (independent of IoU)
        box_format (str): "midpoint" or "corners" to specify the format of the bounding boxes

    returns:
        list: list of bounding boxes after performing NMS with a specific IoU threshold
    """

    # check the data type of the input params
    assert type(bboxes) == list

    # Filter predicted bouding boxes based on probality threshold
    bboxes = [box for box in bboxes if box[1] > threshold]

    # sort the bounding boxes by prob in descending order
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)

    # list to store bouding boxes after NMS
    bboxes_after_nms = []

    # continue loop until the list of boudning box is empty
    while bboxes:
        # get the bouding box with the highest prob
        chosen_box = bboxes.pop(0)

        # remove bounding boxes with IoU greater than the specific threshold with the chosen box

        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0]
            or intersection_over_union(
                torch.tensor(chosen_box[2:]),
                torch.tensor(box[2:]),
                box_format=box_format,
            )
            < iou_threshold
        ]

        # append the chosen boundng box on the list
        bboxes_after_nms.append(chosen_box)

    # return the list of bounding box after NMS
    return bboxes_after_nms

In [ ]:
def mean_average_precision(
        pred_boxes, true_boxes, iou_threshold=0.5, box_format="midpoint", num_classes=20
):
    """
    calculate the mean average precision (mAP)

    params:
        pred_boxes (list): A list of containing predicted bounding boxes with each box define as [train_idx, class_prediction, prob_score, x1, y2, x2, y2]
        true_boxes (list): similar to pred_boxes but containing information about true boxes
        iou_threshold (float): IoU threshold, where perdicted boxes are cosidered correct
        box_format (Str): "midpoint" or "corners" used to specify the format of the boxes
        num_classes (int): number of classes
    
    returns:
        float: the mAP value across all classes with a specific IoU threshold
    """

    # list to store mAP for each class
    average_precisions = []

    # small epsilon to avoid zero division
    epsilon = 1e-6

    for c in range(num_classes):
        detections = []
        ground_truths = []

        # iterate through all predictions and targets, and only add those belonging to the current class 'c'
        for detection in pred_boxes:
            if detection[1] == c:
                detections.append(detection)

        for true_box in true_boxes:
            if true_box[1] == c:
                ground_truths.append(true_box)

        """
        Find the number of boxes for each training example
        the counter here counts the number of target boxes we have
        for each training example, so if image 0 has 3, image 1 has 5 -> we'll have a dictionary like: amount_bboxes = {0 : 3, 1 : 5}
        """
        amount_bboxes = Counter([gt[0] for gt in ground_truths])

        # we the loop through each key, val in this dictionary and convert it to the following (for the same example):
        # amount_bboxes = {0: torch.tensor([0, 0, 0]), 1: torch.tensor([0, 0, 0, 0, 0])}
        for key, val in amount_bboxes.items():
            amount_bboxes[key] = torch.zeros(val)

        # sort by box probability, index 2 is the probability
        detections.sort(key=lambda x: x[2], reverse=True)
        TP = torch.zeros((len(detections)))
        FP = torch.zeros((len(detections)))

        total_true_bboxes = len(ground_truths)

        # if there are no ground truth boxes for this class, if can be safely skipped 
        if total_true_bboxes == 0:
            continue

        for detection_idx, detection in enumerate(detections):
            # only consider ground truth boxes with the same training index as the prediction
            ground_truth_img = [
                bbox for bbox in ground_truths if bbox[0] == detection[0]
            ]

            num_gts = len(ground_truth_img)
            best_iou = 0
            for idx, gt in enumerate(ground_truth_img):
                iou = intersection_over_union(
                    torch.tensor(detection[3:]),
                    torch.tensor(gt[3:]),
                    box_format=box_format
                )
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx

            if best_iou > iou_threshold:
                # only detect ground truth once
                if amount_bboxes[detection[0]][best_gt_idx] == 0:
                    # true position and mark this bounding box as seen
                    TP[detection_idx] = 1
                    amount_bboxes[detection[0]][best_gt_idx] = 1
                else:
                    FP[detection_idx] = 1

            # If IoU is lower, the detection result is false positive
            else:
                FP[detection_idx] = 1

        TP_cumsum = torch.cumsum(TP, dim=0)
        FP_cumsum = torch.cumsum(TP, dim=0)
        recalls = TP_cumsum / (total_true_bboxes + epsilon)
        precisions = torch.divide(TP_cumsum, (TP_cumsum + FP_cumsum + epsilon))
        precisions = torch.cat((torch.tensor([1]), precisions))
        recalls = torch.cat((torch.tensor([0]), recalls))
        average_precisions.append(torch.trapz(precisions, recalls))
    return sum(average_precisions) / len(average_precisions)

# draw bounding box on an image
def plot_image(image, boxes):
    """
    draw predicted bounding boxes on an image
    """

    im = np.array(image)
    height, width, _ = im.shape
    
    # create a figure and axis
    fig, ax = plt.subplots(1)

    # display the image
    ax.imshow(im)

    # each box is represented as [x_center, y_center, width, height]
    for box in boxes:
        box = box[2:]
        assert len(box) == 4, "there are more than 4 values in box (x, y, w, h)"
        
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2
        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),
            box[2] * width,
            box[3] * height,
            linewidth=1,
            edgecolor="r",
            facecolor="none",
        )
        # add the rectangle to the axis
        ax.add_patch(rect)

    plt.show()

# get predicted and true bounding boxes from the model's output and ground truth data
def get_bboxes(
        loader,
        model,
        iou_threshold,
        threshold,
        pred_format="cell",
        box_format="midpoint",
        device="cuda",
        S=7
):
    all_pred_boxes = []
    all_true_boxes = []

    # ensure model is n evaluation mode before obtaining bounding boxes
    model.eval()
    train_idx = 0
    for batch_idx, (x, label) in enumerate(loader):
        x = x.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            predictions = model(x)

        batch_size = x.shape[0]
        true_bboxes = convert_cellboxes(labels).reshape(batch_size, S * S, -1).tolist()
        bboxes = convert_cellboxes(predictions).reshape(batch_size, S * S, -1).tolist()

        for idx in range(batch_size):
            nms_boxes = non_max_suppresion(
                bboxes[idx],
                iou_threshold=iou_threshold,
                threshold=threshold,
                box_format=box_format,
            )
            for nms_box in nms_boxes:
                all_pred_boxes.append([train_idx] + nms_box)

            for box in true_bboxes[idx]:
                # convert multiple boxes to 0 if predicted
                if box[1] > threshold:
                    all_true_boxes.append([train_idx] + box)
            train_idx += 1

    model.train()
    return all_pred_boxes, all_true_boxes

def get_bboxes_training(
    outputs,
    labels,
    iou_threshold=0.5,
    threshold=0.4,
    box_format="midpoint",
    S=7
):
    all_pred_boxes = []
    all_true_boxes = []

    batch_size = outputs.shape[0]

    # ensure the model is in evaluation mode before obtaining bounding boxes
    train_idx = 0
    true_bboxes = convert_cellboxes(labels).reshape(batch_size, S * S, -1).tolist()

    bboxes = convert_cellboxes(outputs).reshape(batch_size, S * S, -1).tolist()

    for idx in range(batch_size):
        nms_boxes = non_max_suppresion(
            bboxes[idx],
            iou_threshold=iou_threshold,
            threshold=threshold,
            box_format=box_format,
        )

        for nms_box in nms_boxes:
            all_pred_boxes.append([train_idx] + nms_box)

        for box in true_bboxes[idx]:
            # convert multiple boxes to 0 if predicted
            if box[1] > threshold:
                all_true_boxes.append([train_idx] + box)

        train_idx += 1

    return all_pred_boxes, all_true_boxes

def convert_cellboxes(predictions, S=7):
    # convert predictions to CPU for processing
    predictions = predictions.to("cpu")

    # determine the batch size from the predictions shape
    batch_size = predictions.shape[0]

    # reshape predictions to a 3D tensor: [batch size, grid size, grid size, features]
    predictions = predictions.reshape(batch_size, S, S, 30)

    # extract bounding box coordinates for the two boxes predicted for each cell
    bboxes1 = predictions[... , 21:25] # shape [batch size, S, S, 4]
    bboxes2 = predictions[..., 26:30] # shape [batch size, S, S, 4]

    # stack the objectness scores for both boxes and find the box with the higher score
    scores = torch.stack((predictions[..., 20], predictions[..., 25]), dim=-1) # shape: [batch size, S, S, 2]
    best_box = scores.argmax(-1).unsqueeze(-1) # shape [batch size, S, S, 1]

    # use the higher score to select the best bounding box for each cell
    best_boxes = torch.where(best_box == 0, bboxes1, bboxes2) # shape: [batch size, S, S, 4]
    
    # create a grid of cell indices for the x and y coordinates
    cell_indices = torch.arange(S, device=predictions.device).view(1, S, 1).expand(batch_size, S, S).unsqueeze(-1)
    x_indices = cell_indices
    y_indices = cell_indices.permute(0, 2, 1, 3) # permute to align y indices

    # adjust the bounding box coordinates from grid scale to image scale
    x = 1 / S * (best_boxes[..., :1] + x_indices)
    y = 1 / S * (best_boxes[..., 1:2] + y_indices)
    w_h = 1 / S * best_boxes[..., 2:4]

    # concatenate the adjusted bounding box coordinates
    converted_bboxes = torch.cat((x, y, w_h), dim=-1)  # Shape: [batch size, S, S, 4]

    # determine the class with the highest probability for each cell
    predicted_class = predictions[..., :20].argmax(-1).unsqueeze(-1) # shape [batch size, S, S, 1]

    # find the maximum confidence score for the best bounding box i each cell
    best_confidence = torch.max(predictions[..., 20], predictions[..., 25]).unsqueeze(-1) # shape: [batch size, S, S, 1]

    # concatenate the class predictions, confidence scores, and bounding boxes
    converted_preds = torch.cat((predicted_class, best_confidence, converted_bboxes), dim=-1) # shape [batch size, S, S, 6]

    return converted_preds

# save checkpoint
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> saving... ")
    torch.save(state, filename)

# load checkpoint
def load_checkpoint(checkpoint, model, optimizer):
    print("=> loading... ")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

# model architecture

In [ ]:
""" 
Information:
A tuple is constructed as (kernel_size, number_of_filers, stride, padding). "M" respresents max-pooling with a 2x2 pool size and 2x3 kernel. The list is structured according to the data blocksm, and ends with an integer representing the number of repetitions.
"""

# describe convolutional and max-pooling layers, as well as the number of repetiitions for convolutional blocks
architecture_config = [
    (7, 64, 2, 3),  # convolutional block 1
    "M",            # max-pooling layer 1
    (3, 192, 1, 1), # convolutional block 2
    "M",            # max-pooling layer 2
    (1, 128, 1, 0), # convolutional block 3
    (3, 256, 1, 1), # convolutional block 4
    (1, 256, 1, 0), # convolutional block 5
    (3, 512, 1, 1), # convolutional block 6
    "M",            # max-pooling layer 3
    [(1, 256, 1, 0), (3, 512, 1, 1), 4], # convolutional block 7 (repeated 4 times)
    (1, 512, 1, 0), # convolutional block 8
    (3, 1024, 1, 1),# convolutional block 9
    "M",            # max-pooling layer 4
    [(1, 512, 1, 0), (3, 1024, 1, 1), 2], # convolutional block 10 (repeated 2 times)
    (3, 1024, 1, 1),# convolutional block 11
    (3, 1024, 1, 1),# convolutional block 12
    (3, 1024, 1, 1),# convolutional block 13
    (3, 1024, 1, 1),# convolutional block 14
]

# a convolutional block is defined with Conv2d, Batchnorm2d and LeakyReLU layers
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super(CNNBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)
        self.batchnorm = nn.BatchNorm2d(out_channels)
        self.leakyrelu = nn.LeakyReLU(0.1)

    def forward(self, x):
        return self.leakyrelu(self.batchnorm(self.conv(x)))
    
# the YOLOv1 model is defined with convolutional layers and fully connected layers
class Yolov1(nn.Module):
    def __init__(self, in_channels=3, **kwargs):
        super(Yolov1, self).__init__()

        self.architecture = architecture_config
        self.in_channels = in_channels
        self.darknet = self._create_conv_layer(self.architecture)
        self.fcs = self._create_fcs(**kwargs)

    def forward(self, x):
        x = self.darknet(x)
        return self.fcs(torch.flatten(x, start_dim=1))
    
    # create conv layer
    def _create_conv_layer(self, architecture):
        layers = []
        in_channels = self.in_channels

        for x in architecture:
            if type(x) == tuple:
                layer += [
                    CNNBlock(
                        in_channels, x[1], kernel_size=x[0], stride=x[2], padding=x[3]
                    )
                ]
                in_channels = x[1]

            elif type(x) == str:
                layer += [nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))]

            elif type(x) == list:
                conv1 = x[0]
                conv2 = x[1]
                num_repeats = x[2]

                for _ in range(num_repeats):
                    layers += [
                        CNNBlock(
                            in_channels,
                            conv1[1],
                            kernel_size=conv1[0],
                            stride=conv1[2],
                            padding=conv1[3]
                        )
                    ]

                    layer += [
                        CNNBlock(
                            conv1[1],
                            conv2[1],
                            kernel_size = conv2[0],
                            stride=conv2[2],
                            padding=conv2[3]
                        )
                    ]
                    in_channels = conv2[1]

        return nn.Sequential(*layers)
    
    def _create_fcs(self, split_size, num_boxes, num_classes):
        S, B, C = split_size, num_boxes, num_classes

        return nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * S * S, 496),
            nn.Dropout(0.0),
            nn.LeakyReLU(0.1),
            nn.Linear(496, S * S * (C + B * 5))
        )
    

# Loss functions

In [ ]:
class YoloLoss(nn.Module):
    def __init__(self, S=7, B=2, C=20):
        """
        initialize the YOLO loss module

        Params:
        S (int): size of grid (set 7 as default)
        B (int): number of bounding boxes (set 2 as default)
        C (int): number of classes (set 20 as default, for VOC dataset)
        """
        super(YoloLoss, self).__init__()
        self.mse = nn.MSELoss(reduction="sum")
        self.S = S
        self.B = B
        self.C = C
        self.lambda_noobj = 0.5 # weight for no object loss
        self.lambda_coord = 5   # weight for box coordinates loss

    def forward(self, predictions, target):
        """
        Calculate loss
        Params:
        predictions (tensor): output from model
        target (tensor): ground truth labels

        Returns:
        tensor : total loss computed from different components
        """

        predictions = predictions.reshape(-1, self.S, self.S, self.C + self.B*5)

        # calculate IoU for the two predicted bounding boxes with the ground truth
        iou_b1 = intersection_over_union(predictions[..., 21:25], target[..., 21:25])
        iou_b2 = intersection_over_union(predictions[..., 26:30], target[..., 21:25])
        ious = torch.cat([iou_b1.unsqueeze(0), iou_b2.unsqueeze(0)], dim=0)

        # determine the bounding box with the highest IoU for each cell
        _, bestbox = torch.max(ious, dim=0)

        # bestbox indicates which bounding box (first or second) has the highest IoU for each cell
        # If bestbox is 0, the first box has a higher IoU, if it's 1, the second box does

        # create a tensor indicating the presence of an object in each grid cell
        exists_box = target[..., 20].unsqueeze(3) # indicator for object presence grid contains an object

        # calculate the different components of the loss
        box_loss = self.calculate_box_loss(predictions, target, bestbox, exists_box)
        object_loss = self.calculate_box_loss(predictions, target, bestbox, exists_box)
        no_object_loss = self.calculate_no_object_loss(predictions, target, exists_box)
        class_loss = self.calculate_class_loss(predictions, target, exists_box)

        # combine the individual losses into the total loss
        loss = (
            box_loss * self.lambda_coord + object_loss + no_object_loss * self.lambda_noobj + class_loss
        )
        return loss
    
    def calculate_box_loss(self, predictions, target, bestbox, exists_box):
        # select the best bounding box predictions and target based on IoU
        box_predictions = exists_box * (bestbox * predictions[..., 26:30] + (1 - bestbox) * predictions[..., 21:25])
        box_targets = exists_box * target[..., 21:25]

        # transform the width and height predictions
        box_predictions[..., 2:4] = torch.sign(box_predictions[..., 2:4]) * torch.sqrt(
            torch.abs(box_predictions[..., 2:4] + 1e-6)
        )
        box_targets[..., 2:4] = torch.sqrt(box_targets[..., 2:4])

        # calculate the mean squared error between the predictions and the targets
        box_loss = self.mse(torch.flatten(box_predictions, end_dim=-2),
                            torch.flatten(box_targets, end_dim=-2))
        return box_loss
    
    def calculate_no_object_loss(self, predictions, target, exists_box):
        # Calculate the mean squared error for cells with no object
        no_object_loss = self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 20:21], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 20:21], start_dim=1)
        )
        no_object_loss += self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 25:26], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 20:21], start_dim=1)
        )
        return no_object_loss

    def calculate_class_loss(self, predictions, target, exists_box):
        # Calculate the mean squared error for the class predictions
        class_loss = self.mse(torch.flatten(exists_box * predictions[..., :20], end_dim=-2),
                              torch.flatten(exists_box * target[..., :20], end_dim=-2))
        return class_loss


# configuration

In [ ]:
# Set the random seed for reproducibility.
seed = 123
torch.manual_seed(seed)

# Hyperparameters and configurations
# Learning rate for the optimizer.
LEARNING_RATE = 2e-5
# Specify whether to use "cuda" (GPU) or "cpu" for training.
DEVICE = "cuda"
# Originally 64 in the research paper, but using a smaller batch size due to GPU limitations.
BATCH_SIZE = 16
# Number of training epochs.
EPOCHS = 300
# Number of worker processes for data loading.
NUM_WORKERS = 2
# If True, DataLoader will pin memory to transfer data to the GPU faster.
PIN_MEMORY = True
# If False, the training process will not load a pre-trained model.
LOAD_MODEL = False
# Specify the file name for the pre-trained model if LOAD_MODEL is True.
LOAD_MODEL_FILE = "final_yolov1.pth.tar"


# training loop

In [ ]:
WIDTH = 448
HEIGHT = 448

def get_train_transforms():
    return A.Compose([A.OneOf([A.HueSaturationValue(hue_shift_limit=0.2, sat_shift_limit= 0.2, val_shift_limit=0.2, p=0.9),
                      A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.9)],p=0.9),
                      A.ToGray(p=0.01),
                      A.HorizontalFlip(p=0.2),
                      A.VerticalFlip(p=0.2),
                      A.Resize(height=WIDTH, width=HEIGHT, p=1),
                    #   A.Cutout(num_holes=8, max_h_size=64, max_w_size=64, fill_value=0, p=0.5),
                      ToTensorV2(p=1.0)],
                      p=1.0,
                      bbox_params=A.BboxParams(format='yolo', min_area=0, min_visibility=0, label_fields=['labels'])
                      )

def get_valid_transforms():
    return A.Compose([A.Resize(height=WIDTH, width=HEIGHT, p=1.0),
                      ToTensorV2(p=1.0)],
                      p=1.0,
                      bbox_params=A.BboxParams(format='yolo', min_area=0, min_visibility=0, label_fields=['labels'])
                      )

class_mapping = {
    'aeroplane': 0,
    'bicycle': 1,
    'bird': 2,
    'boat': 3,
    'bottle': 4,
    'bus': 5,
    'car': 6,
    'cat': 7,
    'chair': 8,
    'cow': 9,
    'diningtable': 10,
    'dog': 11,
    'horse': 12,
    'motorbike': 13,
    'person': 14,
    'pottedplant': 15,
    'sheep': 16,
    'sofa': 17,
    'train': 18,
    'tvmonitor': 19
}


def train_fn(train_loader, model, optimizer, loss_fn, epoch):
    mean_loss = []
    mean_mAP = []

    total_batches = len(train_loader)
    display_interval = total_batches // 5  # Update after 20% of the total batches.

    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = loss_fn(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # get ....
        pred_boxes, true_boxes = get_bboxes_training(out, y, iou_threshold=0.5, threshold=0.4)
        mAP = mean_average_precision(pred_boxes, true_boxes, iou_threshold=0.5, box_format="midpoint")

        mean_loss.append(loss.item())
        mean_mAP.append(mAP.item())

        if batch_idx % display_interval == 0 or batch_idx == total_batches - 1:
            print(f"Epoch: {epoch:3} \t Iter: {batch_idx:3}/{total_batches:3} \t Loss: {loss.item():3.10f} \t mAP: {mAP.item():3.10f}")

    avg_loss = sum(mean_loss) / len(mean_loss)
    avg_mAP = sum(mean_mAP) / len(mean_mAP)
    print(colored(f"Train \t loss: {avg_loss:3.10f} \t mAP: {avg_mAP:3.10f}", 'green'))

    return avg_mAP

def test_fn(test_loader, model, loss_fn, epoch):
    model.eval()
    mean_loss = []
    mean_mAP = []

    for batch_idx, (x, y) in enumerate(test_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = loss_fn(out, y)

        pred_boxes, true_boxes = get_bboxes_training(out, y, iou_threshold=0.5, threshold=0.4)
        mAP = mean_average_precision(pred_boxes, true_boxes, iou_threshold=0.5, box_format="midpoint")

        mean_loss.append(loss.item())
        mean_mAP.append(mAP.item())

    avg_loss = sum(mean_loss) / len(mean_loss)
    avg_mAP = sum(mean_mAP) / len(mean_mAP)
    print(colored(f"Test \t loss: {avg_loss:3.10f} \t mAP: {avg_mAP:3.10f}", 'yellow'))

    model.train()

    return avg_mAP

# Main function.
def train():
    # Initialize the YOLOv1 model, Adam optimizer, and YOLO loss function.
    model = Yolov1(split_size=7, num_boxes=2, num_classes=20).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = YoloLoss()

    # If requested, load a previously saved model.
    if LOAD_MODEL:
        load_checkpoint(torch.load(LOAD_MODEL_FILE), model, optimizer)

    # Create training and validation datasets with specified parameters.
    train_dataset = CustomVOCDataset(root='./data',
                                     year='2012',
                                     image_set='trainval',
                                     download=False,
                                     class_mapping=class_mapping,
                                     custom_transforms=get_train_transforms()
                                     )

    test_dataset = CustomVOCDataset(root='./data',
                                    year='2012',
                                    image_set='val',
                                    download=False,
                                    class_mapping=class_mapping,
                                    custom_transforms=get_valid_transforms()
                                    )

    # Create DataLoader for training and validation datasets with specified settings.
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=True,
    )

    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=False,
        drop_last=False,
    )

    best_mAP_train = 0
    best_mAP_test = 0

    # Iterate through each epoch and compute predicted and target bounding boxes using the get_bboxes function.
    for epoch in range(EPOCHS):
        train_mAP = train_fn(train_loader, model, optimizer, loss_fn, epoch)
        test_mAP = test_fn(test_loader, model, loss_fn, epoch)

        # Update the best mAP for train and test
        if train_mAP > best_mAP_train:
            best_mAP_train = train_mAP

        if test_mAP > best_mAP_test:
            best_mAP_test = test_mAP

            checkpoint = {
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
            }
            save_checkpoint(checkpoint, filename=LOAD_MODEL_FILE)


    print(colored(f"Best Train mAP: {best_mAP_train:3.10f}", 'green'))
    print(colored(f"Best Test mAP: {best_mAP_test:3.10f}", 'yellow'))

In [ ]:
train()

# evaluate model

In [ ]:
LOAD_MODEL = True

def plot_image_with_labels(image, ground_truth_boxes, predicted_boxes, class_mapping):
    """Draw both ground truth and predicted bounding boxes on an image, with labels."""

    # Inverting the class mapping for easy access of class names based on indices
    inverted_class_mapping = {v: k for k, v in class_mapping.items()}

    # Convert the image to a numpy array and get its dimensions
    im = np.array(image)
    height, width, _ = im.shape

    # Create a figure and axis for plotting
    fig, ax = plt.subplots(1)
    # Display the image
    ax.imshow(im)

    # Plot each ground truth box in green
    for box in ground_truth_boxes:
        # Extract label index and bounding box coordinates
        label_index, box = box[0], box[2:]
        # Calculate upper left coordinates
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2
        # Create a rectangle patch
        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),
            box[2] * width,
            box[3] * height,
            linewidth=1,
            edgecolor="green",
            facecolor="none",
        )
        # Add the rectangle to the plot
        ax.add_patch(rect)
        # Retrieve the class name and add it as text to the plot
        class_name = inverted_class_mapping.get(label_index, "Unknown")
        ax.text(upper_left_x * width, upper_left_y * height, class_name, color='white', fontsize=12, bbox=dict(facecolor='green', alpha=0.2))

    # Plot each predicted box in red
    for box in predicted_boxes:
        # Similar processing as for ground truth boxes
        label_index, box = box[0], box[2:]
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2
        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),
            box[2] * width,
            box[3] * height,
            linewidth=1,
            edgecolor="r",
            facecolor="none",
        )
        ax.add_patch(rect)
        class_name = inverted_class_mapping.get(label_index, "Unknown")
        ax.text(upper_left_x * width, upper_left_y * height, class_name, color='white', fontsize=12, bbox=dict(facecolor='red', alpha=0.2))

    plt.show()


def test():
    # Create a YOLO model object with specific hyperparameters.
    model = Yolov1(split_size=7, num_boxes=2, num_classes=20).to(DEVICE)

    # Load saved model weights and optimizer information from a file, if applicable.
    if LOAD_MODEL:
        model.load_state_dict(torch.load(LOAD_MODEL_FILE)['state_dict'])

    # Prepare the test dataset and DataLoader for model evaluation
    test_dataset = CustomVOCDataset(root='./data', image_set='val', download=False)
    test_dataset.init_config_yolo(class_mapping=class_mapping, custom_transforms=get_valid_transforms())
    test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE,
                             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, shuffle=False, drop_last=False)

    model.eval()
    # Iterate over the test dataset and process each batch
    for x, y in test_loader:
        x = x.to(DEVICE)
        out = model(x)

        # Convert model output to bounding boxes and apply non-max suppression
        pred_bboxes = convert_cellboxes(labels).reshape(batch_size, S * S, -1).tolist()
        gt_bboxes = convert_cellboxes(labels).reshape(batch_size, S * S, -1).tolist()

        # Plot the first 8 images with their ground truth and predicted bounding boxes
        for idx in range(8):
            pred_box = non_max_suppression(pred_bboxes[idx], iou_threshold=0.5, threshold=0.4, box_format="midpoint")
            gt_box = non_max_suppression(gt_bboxes[idx], iou_threshold=0.5, threshold=0.4, box_format="midpoint")

            image = x[idx].permute(1,2,0).to("cpu")/255
            plot_image_with_labels(image, gt_box, pred_box, class_mapping)

        break  # Stop after processing the first batch

test()